# Checkpoint sample progression

In [1]:
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from gen_cats.evaluation.experiment_artifacts import catalog_checkpoints, join_fid_checkpoints
from gen_cats.evaluation.report_analysis import (
    best_runs_table,
    ensure_plots_dir,
    load_fid_scores,
    write_stats_csv,
)

plt.rcParams["axes.grid"] = False


PLOTS = ensure_plots_dir("results")
catalog = catalog_checkpoints()
joined = join_fid_checkpoints()
best = {r["model"]: r for r in best_runs_table(load_fid_scores())}
catalog.groupby("model_type")[["n_epoch_samples", "has_samples_best"]].sum().astype(int)

,n_epoch_samples,has_samples_best
model_type,,
beta_vae,131,12
ddim,132,12
pixelcnn,91,6
sn_gan,234,24
tiny_ldm,230,19
vqvae,797,24
wgan_gp,222,24


In [2]:
MAX_FRAMES = 4
SAMPLE_ROWS = 4
SAMPLES_PER_ROW = 4
EPOCH_COLS = 4
PROGRESSION_ORDER = ["wgan_gp", "beta_vae", "vqvae", "sn_gan", "pixelcnn"]


def select_epoch_pngs(slug_dir: Path, max_frames: int = MAX_FRAMES) -> list[Path]:
    epoch_pngs = sorted(
        slug_dir.glob("samples_epoch*.png"),
        key=lambda p: int(p.stem.replace("samples_epoch", "")),
    )
    if not epoch_pngs:
        return []
    if len(epoch_pngs) > max_frames:
        idx = [round(i * (len(epoch_pngs) - 1) / (max_frames - 1)) for i in range(max_frames)]
        epoch_pngs = [epoch_pngs[i] for i in idx]
    return epoch_pngs


def split_sample_grid(path: Path, grid: int = 4) -> list[np.ndarray]:
    img = mpimg.imread(path)
    h, w = img.shape[:2]
    crop = (min(h, w) // grid) * grid
    off_y = (h - crop) // 2
    off_x = (w - crop) // 2
    img = img[off_y : off_y + crop, off_x : off_x + crop]
    tile_h, tile_w = crop // grid, crop // grid
    tiles: list[np.ndarray] = []
    for row in range(grid):
        for col in range(grid):
            tiles.append(img[row * tile_h : (row + 1) * tile_h, col * tile_w : (col + 1) * tile_w])
    return tiles


def stitch_tiles(tiles: list[np.ndarray], rows: int, cols: int, gap: int = 2) -> np.ndarray:
    tile_h, tile_w = tiles[0].shape[:2]
    channels = tiles[0].shape[2] if tiles[0].ndim == 3 else 1
    canvas = np.ones(
        (
            rows * tile_h + (rows - 1) * gap,
            cols * tile_w + (cols - 1) * gap,
            channels,
        ),
        dtype=tiles[0].dtype,
    )
    for row in range(rows):
        for col in range(cols):
            tile_i = row * cols + col
            if tile_i >= len(tiles):
                continue
            y0 = row * (tile_h + gap)
            x0 = col * (tile_w + gap)
            canvas[y0 : y0 + tile_h, x0 : x0 + tile_w] = tiles[tile_i]
    return canvas


def resolve_slug_dir(model: str, slug: str) -> Path | None:
    sub = joined[(joined["model"] == model) & (joined["slug"] == slug)]
    if sub.empty:
        sub = catalog[(catalog["model_type"] == model) & (catalog["slug"] == slug)]
    if sub.empty:
        return None
    return Path(sub.iloc[0]["dir"])


progress_rows = []
panel_data: list[tuple[str, list[Path], str]] = []

for model in PROGRESSION_ORDER:
    row = best.get(model)
    if not row or not row.get("slug"):
        continue
    slug = row["slug"]
    slug_dir = resolve_slug_dir(model, slug)
    if slug_dir is None:
        continue
    epoch_pngs = select_epoch_pngs(slug_dir)
    if not epoch_pngs:
        continue
    panel_data.append((row["label"], epoch_pngs, slug))
    sub = joined[(joined["model"] == model) & (joined["slug"] == slug)]
    if sub.empty:
        sub = catalog[(catalog["model_type"] == model) & (catalog["slug"] == slug)]
    progress_rows.append(
        {
            "model": model,
            "slug": slug,
            "n_epoch_samples": int(sub.iloc[0]["n_epoch_samples"]),
            "figure": "progression_all_best_fid.png",
        }
    )

n_models = len(panel_data)
epoch_rows_per_model = (MAX_FRAMES + EPOCH_COLS - 1) // EPOCH_COLS
fig_width = 10.5
panel_width = fig_width / EPOCH_COLS
model_title_rows = n_models
grid_rows = model_title_rows + n_models * epoch_rows_per_model
fig_height = n_models * (0.12 + epoch_rows_per_model * panel_width) + 0.02 * (n_models - 1)

height_ratios: list[float] = []
for _ in range(n_models):
    height_ratios.append(0.10)
    height_ratios.extend([1.0] * epoch_rows_per_model)

fig = plt.figure(figsize=(fig_width, fig_height))
gs = fig.add_gridspec(
    len(height_ratios), EPOCH_COLS, height_ratios=height_ratios, hspace=0.12, wspace=0.03
)

row_ptr = 0
for label, epoch_pngs, _slug in panel_data:
    title_ax = fig.add_subplot(gs[row_ptr, :])
    title_ax.axis("off")
    title_ax.text(0.0, 0.5, label, ha="left", va="center", fontsize=11, fontweight="bold")
    row_ptr += 1

    for epoch_i, path in enumerate(epoch_pngs):
        row = row_ptr + epoch_i // EPOCH_COLS
        col = epoch_i % EPOCH_COLS
        ax = fig.add_subplot(gs[row, col])
        tiles = split_sample_grid(path)[: SAMPLE_ROWS * SAMPLES_PER_ROW]
        ax.imshow(stitch_tiles(tiles, SAMPLE_ROWS, SAMPLES_PER_ROW, gap=1), aspect="equal")
        ax.set_title(path.stem.replace("samples_", ""), fontsize=8, pad=2, loc="left")
        ax.axis("off")

    for j in range(len(epoch_pngs), epoch_rows_per_model * EPOCH_COLS):
        row = row_ptr + j // EPOCH_COLS
        col = j % EPOCH_COLS
        fig.add_subplot(gs[row, col]).axis("off")

    row_ptr += epoch_rows_per_model

fig.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01)
combined_out = PLOTS / "progression_all_best_fid.png"
fig.savefig(combined_out, dpi=200, bbox_inches="tight", pad_inches=0.01)
plt.close(fig)

write_stats_csv(pd.DataFrame(progress_rows), "progression_runs.csv")
combined_out

PosixPath('/Users/pawelp/Desktop/education/pw/deepl/gen-cats/notebooks/plots/results/progression_all_best_fid.png')